In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

glof = pd.read_csv("../build_ml_GLOF.csv")
glof.head()

,lake_id,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,area_change_km2,peak_discharge_m3s
0,GLO_80.85718_29.91975,2024,0.004988,4782.9,0.36469,0.003,0.0,0.0609,15.00,1.4567,0.000000,121.69
1,GLO_80.87763_30.04078,2017,0.055611,4448.8,1.25819,0.010,0.0,1.1127,15.99,1.5051,0.000000,722.82
2,GLO_80.87763_30.04078,2018,0.052037,4448.8,1.23996,0.010,0.0,1.0261,15.83,1.5334,-0.003574,684.41
3,GLO_80.87763_30.04078,2019,0.055860,4448.8,1.27643,0.010,0.0,1.1188,16.00,1.5235,0.000249,725.49
4,GLO_80.87763_30.04078,2020,0.054780,4448.8,1.22172,0.010,0.0,1.0925,15.95,1.4725,-0.000831,713.93


### Cleaning

In [3]:
glof = glof.sort_values(["lake_id", "year"]).reset_index(drop=True)
glof.head()

,lake_id,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,area_change_km2,peak_discharge_m3s
0,GLO_80.85718_29.91975,2024,0.004988,4782.9,0.36469,0.003,0.0,0.0609,15.00,1.4567,0.000000,121.69
1,GLO_80.87763_30.04078,2017,0.055611,4448.8,1.25819,0.010,0.0,1.1127,15.99,1.5051,0.000000,722.82
2,GLO_80.87763_30.04078,2018,0.052037,4448.8,1.23996,0.010,0.0,1.0261,15.83,1.5334,-0.003574,684.41
3,GLO_80.87763_30.04078,2019,0.055860,4448.8,1.27643,0.010,0.0,1.1188,16.00,1.5235,0.000249,725.49
4,GLO_80.87763_30.04078,2020,0.054780,4448.8,1.22172,0.010,0.0,1.0925,15.95,1.4725,-0.000831,713.93


#### calculating the latitude and longitude from the data
#### this will come in handy later on, when we have to see info on individual lakes

In [4]:
parts = glof["lake_id"].str.split("_", expand=True)
glof["lon"] = parts[1].astype(float)
glof["lat"] = parts[2].astype(float)
glof[["lake_id", "lon", "lat"]].head()

,lake_id,lon,lat
0,GLO_80.85718_29.91975,80.85718,29.91975
1,GLO_80.87763_30.04078,80.87763,30.04078
2,GLO_80.87763_30.04078,80.87763,30.04078
3,GLO_80.87763_30.04078,80.87763,30.04078
4,GLO_80.87763_30.04078,80.87763,30.04078


### Engineered features

In [5]:
# - mean_depth: volume relative to area
# - volume_dam: how much water sits behind how high a dam
# - volume_gap: stored volume vs a simple area*height box
# - uncertainty_frac: measurement noise relative to lake size
# - expansion_frac: growth relative to current area

glof["mean_depth"] = glof["lake_volume_1e6_m3"] / glof["lake_area_km2"]
glof["volume_dam"] = glof["lake_volume_1e6_m3"] * glof["dam_height_m"]
glof["volume_gap"] = glof["lake_volume_1e6_m3"] - glof["lake_area_km2"] * glof["dam_height_m"]
glof["uncertainty_frac"] = glof["area_uncertainty_km2"] / glof["lake_area_km2"]
glof["expansion_frac"] = glof["expansion_rate_km2yr"] / glof["lake_area_km2"]

glof[["mean_depth", "volume_dam", "volume_gap", "uncertainty_frac", "expansion_frac"]].head()

,mean_depth,volume_dam,volume_gap,uncertainty_frac,expansion_frac
0,12.209302,0.913500,-0.013920,0.601443,0.0
1,20.008631,17.792073,0.223480,0.179821,0.0
2,19.718662,16.243163,0.202354,0.192171,0.0
3,20.028643,17.900800,0.225040,0.179019,0.0
4,19.943410,17.425375,0.218759,0.182548,0.0


## Fixing leakage

Primarily we have three leakages

- Q is almost a formula of volume (and dam). keep volume in the data, but the volume rule is the baseline, not a hidden feature. fit that rule on train lakes only.
- the same lake appears in several years. split by lake_id, not by row.
- area_change is 0 on a lake's first year. we do not use it as a feature.


In [ ]:
# random split is problematic

from sklearn.model_selection import train_test_split

idx_tr, idx_te = train_test_split(glof.index, test_size=0.25, random_state=42)
overlap = set(glof.loc[idx_tr, "lake_id"]) & set(glof.loc[idx_te, "lake_id"])
print("random split, lakes on both sides:", len(overlap))

random split, lakes on both sides: 1438


#### since same lake appears twice, there are chances that the model will see the same lake in test phase as well

There are 1,438 lakes appearing in both training and testing sets.

That is a significant leakage problem.

In [12]:
# splitting by lake which is our solution

lakes = glof["lake_id"].drop_duplicates().to_numpy()
np.random.seed(42)
np.random.shuffle(lakes)
cut = int(0.75 * len(lakes))
train_lakes = lakes[:cut]
test_lakes = lakes[cut:]

train = glof[glof["lake_id"].isin(train_lakes)].copy()
test = glof[glof["lake_id"].isin(test_lakes)].copy()

print("train rows", len(train), "test rows", len(test))
print("train lakes", train["lake_id"].nunique(), "test lakes", test["lake_id"].nunique())
print("lakes on both sides:", len(set(train["lake_id"]) & set(test["lake_id"])))

train rows 8799 test rows 3105
train lakes 1746 test lakes 583
lakes on both sides: 0


In [13]:
# calculating the volume rule, establishing the baseline
#  this is the number a later model has to beat

V_tr = train["lake_volume_1e6_m3"].to_numpy()
Q_tr = train["peak_discharge_m3s"].to_numpy()
V_te = test["lake_volume_1e6_m3"].to_numpy()
Q_te = test["peak_discharge_m3s"].to_numpy()

X = np.column_stack([np.log(V_tr), np.ones(len(V_tr))])
coef, *_ = np.linalg.lstsq(X, np.log(Q_tr), rcond=None)
a = np.exp(coef[1])
b = coef[0]
pred_te = a * V_te ** b

print(f"train fit: Q ≈ {a:.2f} * V^{b:.3f}")
print("test mae, volume rule:", np.mean(np.abs(Q_te - pred_te)))
print("test mae, predict train median:", np.mean(np.abs(Q_te - np.median(Q_tr))))

train fit: Q ≈ 730.60 * V^0.642
test mae, volume rule: 122.47723987166351
test mae, predict train median: 578.2296747181965


In [9]:
# - columns we will actually use later. target stays out. volume rule stays a baseline.

FEATURES = [
    "year",
    "lake_area_km2",
    "elevation_m",
    "perimeter_km",
    "area_uncertainty_km2",
    "expansion_rate_km2yr",
    "lake_volume_1e6_m3",
    "dam_height_m",
    "shape_index",
    "lon",
    "lat",
    "mean_depth",
    "volume_gap",
    "uncertainty_frac",
    "expansion_frac",
]
TARGET = "peak_discharge_m3s"

# volume_dam left out — it is almost the same as the volume rule
# area_change left out — zero on first year, not a real yearly growth

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("target in features:", TARGET in FEATURES)
print(X_train.shape, X_test.shape)
X_train.head()

target in features: False
(8799, 15) (3105, 15)


,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,lon,lat,mean_depth,volume_gap,uncertainty_frac,expansion_frac
0,2024,0.004988,4782.9,0.36469,0.003,0.0000,0.0609,15.0,1.4567,80.85718,29.91975,12.209302,-0.013920,0.601443,0.000000
5,2017,0.030424,4371.9,0.91173,0.007,0.0005,0.5285,15.0,1.4745,80.88324,30.05350,17.371154,0.072140,0.230082,0.016434
6,2018,0.031837,4371.9,1.00291,0.008,0.0005,0.5586,15.0,1.5856,80.88324,30.05350,17.545623,0.081045,0.251280,0.015705
7,2019,0.032419,4371.9,1.03938,0.008,0.0005,0.5711,15.0,1.6284,80.88324,30.05350,17.616213,0.084815,0.246769,0.015423
8,2020,0.035412,4371.9,1.13055,0.008,0.0005,0.6360,15.0,1.6948,80.88324,30.05350,17.960014,0.104820,0.225912,0.014120


In [10]:
glof.to_csv("glof_clean.csv", index=False)
print(glof.shape)
glof.head()

(11904, 19)


,lake_id,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,area_change_km2,peak_discharge_m3s,lon,lat,mean_depth,volume_dam,volume_gap,uncertainty_frac,expansion_frac
0,GLO_80.85718_29.91975,2024,0.004988,4782.9,0.36469,0.003,0.0,0.0609,15.00,1.4567,0.000000,121.69,80.85718,29.91975,12.209302,0.913500,-0.013920,0.601443,0.0
1,GLO_80.87763_30.04078,2017,0.055611,4448.8,1.25819,0.010,0.0,1.1127,15.99,1.5051,0.000000,722.82,80.87763,30.04078,20.008631,17.792073,0.223480,0.179821,0.0
2,GLO_80.87763_30.04078,2018,0.052037,4448.8,1.23996,0.010,0.0,1.0261,15.83,1.5334,-0.003574,684.41,80.87763,30.04078,19.718662,16.243163,0.202354,0.192171,0.0
3,GLO_80.87763_30.04078,2019,0.055860,4448.8,1.27643,0.010,0.0,1.1188,16.00,1.5235,0.000249,725.49,80.87763,30.04078,20.028643,17.900800,0.225040,0.179019,0.0
4,GLO_80.87763_30.04078,2020,0.054780,4448.8,1.22172,0.010,0.0,1.0925,15.95,1.4725,-0.000831,713.93,80.87763,30.04078,19.943410,17.425375,0.218759,0.182548,0.0
